# Job Hunting

Find hiring companies, filter by your preferences, and build shortlists.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

conn = sqlite3.connect("../data/yc_collaboration.db")
df = pd.read_sql("SELECT * FROM companies", conn)
conn.close()

df["batch_year"] = df["batch"].str.extract(r"(\d{4})").astype(float)
df["city"] = df["location"].str.split(",").str[0].str.strip()

# Base: active and hiring
hiring = df[(df["is_hiring"] == 1) & (df["status"] == "Active")].copy()
print(f"Active & hiring: {len(hiring)} companies out of {len(df)} total")

---
## 1. Hiring Snapshot

In [ ]:
print(f"{'Active & hiring':<25} {len(hiring):,}")
print(f"{'Median team size':<25} {int(hiring['team_size'].median())}")
print(f"{'Mean team size':<25} {int(hiring['team_size'].mean())}")
print(f"{'Early stage':<25} {(hiring['stage'] == 'Early').sum()}")
print(f"{'Growth stage':<25} {(hiring['stage'] == 'Growth').sum()}")
print()
print("=== By Industry ===")
print(hiring["industry"].value_counts().to_string())
print()
print("=== By Stage ===")
print(hiring["stage"].value_counts().to_string())

---
## 2. Hiring by Location

In [ ]:
# Has remote in location or regions?
hiring["is_remote"] = (
    hiring["location"].str.contains("Remote", case=False, na=False) |
    hiring["regions"].str.contains("Remote", case=False, na=False)
)

remote_count = hiring["is_remote"].sum()
print(f"Remote-friendly: {remote_count} ({remote_count/len(hiring)*100:.1f}%)")
print()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

hiring["city"].value_counts().head(15).plot(kind="barh", ax=axes[0], color="#333333", edgecolor="white")
axes[0].set_title("Top 15 Cities (Hiring)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Count")
axes[0].invert_yaxis()

regions = hiring["regions"].str.split("; ").explode().str.strip()
regions = regions[(regions != "") & (~regions.str.contains("Remote", na=False))]
regions.value_counts().head(10).plot(kind="barh", ax=axes[1], color="#333333", edgecolor="white")
axes[1].set_title("Top 10 Regions (Hiring)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Count")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 3. Hiring by Batch Year (Company Age)

In [ ]:
batch_hiring = hiring["batch_year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
batch_hiring.plot(kind="bar", ax=ax, color="#333333", edgecolor="white")
ax.set_title("Hiring Companies by Batch Year", fontsize=14, fontweight="bold")
ax.set_xlabel("Batch Year")
ax.set_ylabel("Hiring Companies")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nMost hiring companies come from recent batches (newer startups growing fast).")

---
## 4. Hiring by Team Size Bucket

In [ ]:
bins = [0, 5, 10, 25, 50, 100, 250, 500, 1000, 50000]
labels = ["1-5", "6-10", "11-25", "26-50", "51-100", "101-250", "251-500", "501-1K", "1K+"]
hiring["size_bucket"] = pd.cut(hiring["team_size"], bins=bins, labels=labels)

bucket_counts = hiring["size_bucket"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
bucket_counts.plot(kind="bar", ax=ax, color="#333333", edgecolor="white")
ax.set_title("Hiring Companies by Team Size", fontsize=14, fontweight="bold")
ax.set_xlabel("Team Size")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print()
for label, count in bucket_counts.items():
    print(f"  {label:<10} {count:>5} companies")

---
## 5. Top Tags Among Hiring Companies

In [ ]:
hiring_tags = hiring["tags"].str.split("; ").explode().str.strip()
hiring_tags = hiring_tags[(hiring_tags != "") & hiring_tags.notnull()]

fig, ax = plt.subplots(figsize=(10, 8))
hiring_tags.value_counts().head(20).plot(kind="barh", ax=ax, color="#333333", edgecolor="white")
ax.set_title("Top 20 Tags (Hiring Companies Only)", fontsize=14, fontweight="bold")
ax.set_xlabel("Count")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 6. Remote-Friendly Hiring Companies

In [ ]:
remote = hiring[hiring["is_remote"]].sort_values("team_size", ascending=False)

print(f"Remote-friendly & hiring: {len(remote)} companies\n")

print("=== Top 25 by Team Size ===")
remote[["name", "one_liner", "website", "team_size", "industry", "batch"]].head(25)

---
## 7. Recently Founded & Hiring (High Growth Potential)

In [ ]:
recent_hiring = hiring[hiring["batch_year"] >= 2023].sort_values("team_size", ascending=False)

print(f"Founded 2023+ & hiring: {len(recent_hiring)} companies\n")

print("=== Top 30 Newest Hiring Companies ===")
recent_hiring[["name", "one_liner", "website", "team_size", "batch", "location"]].head(30)

---
## 8. Established & Hiring (Stability)

In [ ]:
established = hiring[(hiring["batch_year"] <= 2020) & (hiring["team_size"] >= 50)].sort_values("team_size", ascending=False)

print(f"Established (pre-2020, 50+ team) & hiring: {len(established)} companies\n")

print("=== Top 30 Established Hiring Companies ===")
established[["name", "one_liner", "website", "team_size", "batch", "location"]].head(30)

---
## 9. Search by Skills / Keywords

Change the keyword to match your skills.

In [ ]:
def job_search(
    keyword=None,
    industry=None,
    min_team=0,
    max_team=100000,
    remote_only=False,
    min_batch_year=None,
    max_batch_year=None,
    stage=None,
):
    """Search hiring companies by your criteria."""
    result = hiring.copy()
    if keyword:
        mask = (
            result["name"].str.contains(keyword, case=False, na=False)
            | result["one_liner"].str.contains(keyword, case=False, na=False)
            | result["long_description"].str.contains(keyword, case=False, na=False)
            | result["tags"].str.contains(keyword, case=False, na=False)
        )
        result = result[mask]
    if industry:
        result = result[result["industry"].str.contains(industry, case=False, na=False)]
    if remote_only:
        result = result[result["is_remote"]]
    if min_batch_year:
        result = result[result["batch_year"] >= min_batch_year]
    if max_batch_year:
        result = result[result["batch_year"] <= max_batch_year]
    if stage:
        result = result[result["stage"] == stage]
    result = result[(result["team_size"] >= min_team) & (result["team_size"] <= max_team)]
    result = result.sort_values("team_size", ascending=False)
    print(f"Found: {len(result)} companies\n")
    return result[["name", "one_liner", "website", "team_size", "batch", "location", "industry", "tags"]]


# === EDIT THESE TO MATCH YOUR SKILLS ===

# Example: Backend / Python roles
job_search(keyword="python", remote_only=False).head(20)

In [ ]:
# Example: AI companies, 10-200 people, remote
job_search(keyword="AI", min_team=10, max_team=200, remote_only=True).head(20)

In [ ]:
# Example: B2B SaaS, Growth stage
job_search(keyword="SaaS", industry="B2B", stage="Growth").head(20)

In [ ]:
# Example: Developer tools
job_search(keyword="Developer Tools").head(20)

---
## 10. Export Shortlist

Save your filtered results to a CSV for tracking applications.

In [ ]:
# Run your search and save
my_shortlist = job_search(keyword="AI", min_team=5, remote_only=True)

# Save to CSV
output_path = "../data/my_job_shortlist.csv"
my_shortlist.to_csv(output_path, index=False)
print(f"Saved {len(my_shortlist)} companies to {output_path}")